# 01 | Data Preparation

This notebook loads the raw dataset, refreshes the cached summary if needed, and checks the basic fields used later in the project.

Before running it, download the raw dataset from `https://networkrepository.com/rec-dating.php`
and place the file at `data/rec-dating/rec-dating.edges`.


## Workflow

Run the notebooks in this order:

1. **Data preparation**: inspect the raw file and cache the dataset summary
2. **Exploration**: review the main descriptive tables and plots
3. **Applications**: run the concentration and feature-alignment analyses
4. **Final plots**: collect the last figures and reference values


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "rec_dating_project").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


project_root = locate_project_root()
sys.path.insert(0, str(project_root / "src"))

from rec_dating_project import ProjectPaths, RecDatingDataset


paths = ProjectPaths.default()
paths.ensure_output_dirs()
OUTPUT_DATA = paths.output_data_dir
FORCE_REBUILD = False


def run_script(script_name: str, *args: str) -> None:
    cmd = [sys.executable, str(paths.scripts_dir / script_name), *args]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=paths.project_root)


def ensure_dataset_summary(force: bool = False) -> Path:
    summary_path = OUTPUT_DATA / "dataset_summary.json"
    if force or not summary_path.exists():
        run_script("01_dataset_overview.py")
    return summary_path


summary_path = ensure_dataset_summary(FORCE_REBUILD)
summary = json.loads(summary_path.read_text(encoding="utf-8"))
dataset = RecDatingDataset(paths.raw_edges_path)

print(f"Project root: {paths.project_root}")
print(f"Raw edge file: {paths.raw_edges_path}")
print(f"Summary cache: {summary_path}")


## Raw File Preview

The raw edge file has exactly three columns and no header row:

- `rater_id`
- `profile_id`
- `rating`

Each row is one directed rating event from a rater to a profile.


In [ ]:
raw_preview = dataset.read_edges(nrows=8)
raw_preview


In [ ]:
field_dictionary = pd.DataFrame(
    [
        {
            "field": "rater_id",
            "description": "User ID on the sending side of the edge.",
        },
        {
            "field": "profile_id",
            "description": "User ID on the receiving side of the edge.",
        },
        {
            "field": "rating",
            "description": "Observed score on the 1-10 scale.",
        },
    ]
)

display(field_dictionary)


## Basic Dataset Scale

Before we do any network analysis, we want a compact picture of the dataset size and the score distribution.

This summary is cached in `outputs/data/dataset_summary.json`, so later notebooks can reuse it without re-reading the full file every time.


In [ ]:
summary_table = pd.DataFrame(
    [
        {"metric": "edge_count", "value": summary["edge_count"]},
        {"metric": "unique_raters", "value": summary["unique_raters"]},
        {"metric": "unique_profiles", "value": summary["unique_profiles"]},
        {"metric": "unique_users_union", "value": summary["unique_users_union"]},
        {"metric": "overlapping_user_ids", "value": summary["overlapping_user_ids"]},
        {"metric": "exclusive_profile_ids", "value": summary["exclusive_profile_ids"]},
        {"metric": "mean_rating", "value": summary["mean_rating"]},
    ]
)

rating_distribution = pd.DataFrame(
    [
        {"rating": int(rating), "count": int(count)}
        for rating, count in summary["rating_histogram"].items()
    ]
).sort_values("rating")
rating_distribution["share"] = rating_distribution["count"] / rating_distribution["count"].sum()

display(summary_table)
display(rating_distribution)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(rating_distribution["rating"], rating_distribution["share"], color="#386641", width=0.8)
ax.set_title("Rating Distribution in the Raw Dataset")
ax.set_xlabel("Rating")
ax.set_ylabel("Share of all edges")
ax.set_xticks(rating_distribution["rating"])
ax.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.show()


## Why a Role-Based Bipartite Network?

One modeling choice matters for every later result:

we do **not** treat the same numeric identifier as a single universal node.

Instead, we separate the two structural roles:

- the **rater role**: sending scores
- the **profile role**: receiving scores

That choice keeps the directional meaning of the data intact and avoids unsupported identity assumptions across roles.


In [ ]:
role_table = pd.DataFrame(
    [
        {
            "quantity": "unique_raters",
            "value": summary["unique_raters"],
            "why it matters": "This is the sending side of the network.",
        },
        {
            "quantity": "unique_profiles",
            "value": summary["unique_profiles"],
            "why it matters": "This is the receiving side of the network.",
        },
        {
            "quantity": "overlapping_user_ids",
            "value": summary["overlapping_user_ids"],
            "why it matters": "Many numeric IDs appear in both roles, so the roles must be separated analytically.",
        },
    ]
)

top_raters = pd.DataFrame(summary["top_raters"]).head(10)
top_profiles = pd.DataFrame(summary["top_profiles"]).head(10)

display(role_table)
display(Markdown("### Top raters by outgoing activity"))
display(top_raters)
display(Markdown("### Top profiles by incoming attention"))
display(top_profiles)


In [ ]:
display(
    Markdown(
        f"""
        ## Key Points

        - The dataset is large enough to support full-network analysis: **{summary['edge_count']:,}** edges, **{summary['unique_raters']:,}** raters, and **{summary['unique_profiles']:,}** profiles.
        - The overlap between `rater_id` and `profile_id` values is substantial (**{summary['overlapping_user_ids']:,}** IDs), which justifies the role-based bipartite representation.
        - The score distribution is not flat; the upper end of the scale is used heavily, which matters for the later positive-layer analysis.
        """
    )
)


## Output

The key cached artifact from this notebook is:

- `outputs/data/dataset_summary.json`

The next notebook uses that summary to explore popularity, prestige, and inequality.
